# E3 Recovery — Qwen3-VL-4B trên Colab T4

Notebook này khôi phục môi trường và tiếp tục một **nhánh phục hồi** từ LoRA `step050` trong Drive.

**Quan trọng:** checkpoint gốc `step050` thiếu `extra_trainable.pt` (projector đã train). Vì vậy notebook này giữ LoRA từ step 50, khởi tạo projector từ model gốc, và tạo optimizer/scheduler mới. Đây **không phải resume chính xác** thí nghiệm cũ. Toàn bộ kết quả được lưu vào thư mục mới `E3_recovery_from_step050`; không ghi đè checkpoint gốc.

Chạy lần lượt từng cell. Không cần chạy notebook cũ song song.

## Cell 1 — Chọn GPU và clone repo

In [ ]:
# Colab: Runtime → Change runtime type → T4 GPU
!nvidia-smi -L

from pathlib import Path
import os

REPO = Path('/content/K4-Track-4-Day10-VisionLanguageModel')
if not REPO.exists():
    !git clone https://github.com/dangkhoabui15092004-byte/K4-Track-4-Day10-VisionLanguageModel.git
%cd /content/K4-Track-4-Day10-VisionLanguageModel
!pip -q install -r requirements-colab.txt


## Cell 2 — Mount Drive và xác minh checkpoint trước khi làm gì khác

In [ ]:
from google.colab import drive
from pathlib import Path
import json, os

drive.mount('/content/drive', force_remount=False)
SOURCE_CKPT = Path('/content/drive/MyDrive/ROS_VLM_Checkpoints/E3_safe_run/step050')
OUTPUT_ROOT = Path('/content/drive/MyDrive/ROS_VLM_Checkpoints/E3_recovery_from_step050')

assert SOURCE_CKPT.exists(), f'Không tìm thấy {SOURCE_CKPT}'
for name in ['adapter_config.json','adapter_model.safetensors','training_state.pt','run.json','extra_trainable.pt']:
    p = SOURCE_CKPT / name
    print(f'{name}:', f'{p.stat().st_size/1024**2:.2f} MB' if p.exists() else 'THIẾU')

assert (SOURCE_CKPT/'adapter_config.json').exists()
assert (SOURCE_CKPT/'adapter_model.safetensors').exists()
assert not (SOURCE_CKPT/'extra_trainable.pt').exists(), 'Checkpoint nguồn khác với trạng thái đã kiểm tra; dừng để kiểm tra.'
print('Output mới:', OUTPUT_ROOT)
print('Checkpoint gốc sẽ không bị ghi đè.')


## Cell 3 — Chuẩn bị dữ liệu
Nếu Colab Secrets đã có `KAGGLE_USERNAME` và `KAGGLE_KEY`, chạy cell này. Nếu dataset đã có đủ trong `data/splits/`, cell sẽ bỏ qua tải dữ liệu.

In [ ]:
from pathlib import Path

if all(Path(f'data/splits/{s}.jsonl').exists() for s in ['train','val','test']):
    print('Đã có các file split, bỏ qua tải dataset.')
else:
    from google.colab import userdata
    import os
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    !python scripts/prepare_data.py --kaggle

for s in ['train','val','test']:
    p = Path(f'data/splits/{s}.jsonl')
    assert p.exists(), f'Thiếu {p}; kiểm tra Kaggle Secrets và bước chuẩn bị dữ liệu.'
    print(s, 'OK', p.stat().st_size/1024**2, 'MB')


## Cell 4 — Imports, nhãn và prompt

In [ ]:
import gc, json, random, time, shutil, re
from pathlib import Path
from PIL import Image
import torch
import bitsandbytes as bnb
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration, BitsAndBytesConfig, get_cosine_schedule_with_warmup
from peft import PeftModel

assert torch.cuda.is_available(), 'Chưa bật GPU trong Runtime.'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1024**3,2), 'GiB')

MODEL_ID = 'Qwen/Qwen3-VL-4B-Instruct'
CLASSES = ['pothole','crack','manhole']
PROMPT = (
    'Detect road damage in this dashcam image. Classes: pothole, crack, manhole.\n'
    'Reply with ONLY a JSON list, no other text, no code fences. '
    'Each item: {"label": <class>, "bbox_2d": [x1, y1, x2, y2]}.\n'
    'Coordinates are integers on a 0-1000 scale relative to image width/height (NOT pixels). '
    'If nothing is present, reply [].'
)

def read_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

splits = {s: read_jsonl(f'data/splits/{s}.jsonl') for s in ['train','val','test']}
train_entries = splits['train']
print({k:len(v) for k,v in splits.items()})


## Cell 5 — Bộ chuyển nhãn YOLO, encode và collate

In [ ]:
def yolo_to_box(label, yolo):
    xc, yc, w, h = yolo
    vals = [(xc-w/2)*1000, (yc-h/2)*1000, (xc+w/2)*1000, (yc+h/2)*1000]
    return {'label':label, 'bbox_2d':[max(0,min(1000,round(v))) for v in vals]}

def objects_to_answer(objects):
    boxes = [yolo_to_box(o['label'], o['yolo']) for o in objects]
    boxes.sort(key=lambda b:(b['bbox_2d'][1], b['bbox_2d'][0]))
    return json.dumps(boxes, separators=(',',':'))

IGNORE_INDEX = -100
def encode_example(image, prompt, answer, processor):
    user_msg = [{'role':'user','content':[{'type':'image','image':image},{'type':'text','text':prompt}]}]
    assistant_msg = user_msg + [{'role':'assistant','content':answer}]
    prompt_enc = processor.apply_chat_template(user_msg, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors='pt')
    full_enc = processor.apply_chat_template(assistant_msg, tokenize=True, add_generation_prompt=False, return_dict=True, return_tensors='pt')
    n = prompt_enc['input_ids'].shape[1]
    assert torch.equal(full_enc['input_ids'][0,:n], prompt_enc['input_ids'][0]), 'Chat template không khớp.'
    labels = full_enc['input_ids'].clone()
    labels[:,:n] = IGNORE_INDEX
    full_enc['labels'] = labels
    return full_enc

def collate(batch):
    pad_id = processor.tokenizer.pad_token_id
    text_keys = [k for k in ('input_ids','attention_mask','labels','mm_token_type_ids') if k in batch[0]]
    max_len = max(b['input_ids'].shape[1] for b in batch)
    out = {}
    for k in text_keys:
        pad_value = IGNORE_INDEX if k == 'labels' else (pad_id if k == 'input_ids' else 0)
        rows=[]
        for b in batch:
            t=b[k][0]
            if t.shape[0] < max_len:
                t=torch.cat([t,torch.full((max_len-t.shape[0],),pad_value,dtype=t.dtype)])
            rows.append(t)
        out[k]=torch.stack(rows)
    out['pixel_values']=torch.cat([b['pixel_values'] for b in batch],dim=0)
    out['image_grid_thw']=torch.cat([b['image_grid_thw'] for b in batch],dim=0)
    return out

def sample_batch(entries, batch_size, processor):
    picks = random.sample(entries,batch_size) if len(entries)>=batch_size else random.choices(entries,k=batch_size)
    encs=[]
    for e in picks:
        with Image.open(f"data/{e['image_path']}") as im:
            img=im.convert('RGB')
        encs.append(encode_example(img,PROMPT,objects_to_answer(e['objects']),processor))
    return collate(encs)


## Cell 6 — Tải processor và định nghĩa loader model gốc

In [ ]:
def build_quantization_config():
    return BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
        llm_int8_skip_modules=['visual','lm_head'],
    )

def fresh_base_model():
    return Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID, quantization_config=build_quantization_config(),
        dtype=torch.float16, device_map={'':0},
    )

processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = 'left'
print('Processor sẵn sàng:', type(processor).__name__)


## Cell 7 — Nạp LoRA từ step050 và projector pretrained
Cell này tải model lên GPU. Không train ở cell này; chỉ dựng model và kiểm tra tham số.

In [ ]:
from peft import PeftModel

PROJ_KEYS = ('visual.merger','visual.deepstack_merger_list')

# Dọn GPU trước khi tải model
gc.collect(); torch.cuda.empty_cache()
base = fresh_base_model()
model = PeftModel.from_pretrained(base, str(SOURCE_CKPT), is_trainable=True)

# Checkpoint step050 không có projector đã train. Dùng projector từ model gốc và bật trainable.
for name, p in list(model.named_parameters()):
    if any(k in name for k in PROJ_KEYS):
        mod_name, _, pname = name.rpartition('.')
        mod = model.get_submodule(mod_name)
        setattr(mod, pname, torch.nn.Parameter(p.detach().float().contiguous(), requires_grad=True))

model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant':False})
model.enable_input_require_grads()
model.config.use_cache = False
trainable = [p for p in model.parameters() if p.requires_grad]
print('Trainable params:', sum(p.numel() for p in trainable))
print('Trainable dtypes:', sorted({str(p.dtype) for p in trainable}))
print('VRAM:', round(torch.cuda.memory_allocated()/1024**3,2),'GB')
assert any(any(k in n for k in PROJ_KEYS) and p.requires_grad for n,p in model.named_parameters()), 'Projector chưa được bật trainable.'


## Cell 8 — Recovery training (bắt đầu từ step 51)
Mặc định chạy đến step 300. Để test nhanh, có thể đặt `END_STEP = 51`; sau khi xác nhận chạy được, đổi lại 300 và chạy cell này. Các checkpoint được ghi vào thư mục mới và không ghi đè checkpoint gốc.

In [ ]:
import torch.nn as nn

START_STEP = 50
END_STEP = 300
BATCH_SIZE = 1
GRAD_ACCUM = 8
SAVE_EVERY = 50
LR = 2e-4

# Optimizer/scheduler mới vì projector và optimizer cũ không thể phục hồi nhất quán.
optimizer = bnb.optim.PagedAdamW8bit(trainable, lr=LR, weight_decay=0.01)
remaining_steps = END_STEP - START_STEP
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=min(5, max(0, remaining_steps//10)),
    num_training_steps=max(1, remaining_steps),
)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_ROOT = Path('/content/e3_recovery_checkpoints')
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
losses=[]

def save_recovery_checkpoint(step, latest_loss):
    name=f'step{step:03d}'
    local_tmp=LOCAL_ROOT / (name+'_tmp')
    if local_tmp.exists(): shutil.rmtree(local_tmp)
    local_tmp.mkdir(parents=True)
    model.save_pretrained(local_tmp)
    projector_state={n:p.detach().cpu().clone() for n,p in model.named_parameters()
                     if p.requires_grad and any(k in n for k in PROJ_KEYS)}
    if not projector_state: raise RuntimeError('Không có projector trainable; dừng lưu.')
    torch.save(projector_state,local_tmp/'extra_trainable.pt')
    torch.save({'step':step,'optimizer':optimizer.state_dict(),'scheduler':scheduler.state_dict(),'losses':losses},local_tmp/'training_state.pt')
    metadata={
        'model_id':MODEL_ID,'experiment':'E3_recovery_from_step050',
        'recovery_mode':'LoRA from original step050; projector reset to pretrained base weights; optimizer and scheduler reset',
        'original_step':START_STEP,'step':step,'max_steps':END_STEP,
        'batch_size':BATCH_SIZE,'gradient_accumulation':GRAD_ACCUM,'learning_rate':LR,
        'projector_tensor_count':len(projector_state),'latest_loss':latest_loss,
    }
    (local_tmp/'run.json').write_text(json.dumps(metadata,indent=2),encoding='utf-8')
    required=['adapter_config.json','adapter_model.safetensors','extra_trainable.pt','training_state.pt','run.json']
    missing=[n for n in required if not (local_tmp/n).exists() or (local_tmp/n).stat().st_size==0]
    if missing: raise RuntimeError(f'Checkpoint local chưa đầy đủ: {missing}')
    # Upload vào tên tạm trên Drive; chỉ đổi tên sau khi kiểm tra đủ file. Không xóa checkpoint cũ.
    drive_tmp=OUTPUT_ROOT/(name+'_uploading')
    drive_final=OUTPUT_ROOT/name
    if drive_tmp.exists(): shutil.rmtree(drive_tmp)
    shutil.copytree(local_tmp,drive_tmp)
    missing=[n for n in required if not (drive_tmp/n).exists() or (drive_tmp/n).stat().st_size==0]
    if missing: raise RuntimeError(f'Bản Drive chưa đầy đủ: {missing}')
    if drive_final.exists():
        raise FileExistsError(f'{drive_final} đã tồn tại; không ghi đè checkpoint. Đổi tên run/output root nếu cần.')
    drive_tmp.rename(drive_final)
    print('Checkpoint đã xác minh:',drive_final)
    for p in sorted(drive_final.iterdir()): print(' ',p.name,round(p.stat().st_size/1024**2,2),'MB')

# Lưu trạng thái bắt đầu trước khi train để có bản khôi phục an toàn
if not (OUTPUT_ROOT/'step050_recovery_start').exists():
    start_dir=LOCAL_ROOT/'step050_recovery_start'
    if start_dir.exists(): shutil.rmtree(start_dir)
    start_dir.mkdir(parents=True)
    model.save_pretrained(start_dir)
    projector_state={n:p.detach().cpu().clone() for n,p in model.named_parameters() if p.requires_grad and any(k in n for k in PROJ_KEYS)}
    torch.save(projector_state,start_dir/'extra_trainable.pt')
    torch.save({'step':START_STEP,'optimizer':optimizer.state_dict(),'scheduler':scheduler.state_dict(),'losses':[]},start_dir/'training_state.pt')
    (start_dir/'run.json').write_text(json.dumps({'model_id':MODEL_ID,'experiment':'E3_recovery_from_step050','step':START_STEP,'recovery_mode':'LoRA step050 + pretrained projector; fresh optimizer'},indent=2),encoding='utf-8')
    start_drive=OUTPUT_ROOT/'step050_recovery_start'
    shutil.copytree(start_dir,start_drive,dirs_exist_ok=False)
    print('Đã lưu trạng thái bắt đầu:',start_drive)

model.train()
optimizer.zero_grad(set_to_none=True)
torch.cuda.reset_peak_memory_stats()
start_time=time.time()
for step in range(START_STEP+1, END_STEP+1):
    step_loss=0.0
    for micro in range(GRAD_ACCUM):
        batch=sample_batch(train_entries,BATCH_SIZE,processor)
        batch={k:v.to('cuda') if isinstance(v,torch.Tensor) else v for k,v in batch.items()}
        with torch.autocast(device_type='cuda',dtype=torch.float16):
            out=model(**batch)
            loss=out.loss/GRAD_ACCUM
        if not torch.isfinite(loss): raise RuntimeError(f'Loss không hữu hạn tại step {step}')
        loss.backward()
        step_loss += loss.detach().float().item()
        del batch,out,loss
    torch.nn.utils.clip_grad_norm_(trainable,1.0)
    optimizer.step(); scheduler.step(); optimizer.zero_grad(set_to_none=True)
    losses.append(step_loss)
    if step==START_STEP+1 or step%10==0:
        print(f'Step {step}/{END_STEP} | loss={step_loss:.4f} | VRAM={torch.cuda.memory_allocated()/1024**3:.2f} GB | elapsed={(time.time()-start_time)/60:.1f} min')
    if step%SAVE_EVERY==0 or step==END_STEP:
        save_recovery_checkpoint(step,step_loss)

model.config.use_cache=True
print('Hoàn tất nhánh E3 recovery. Đây không phải resume chính xác trạng thái E3 ban đầu.')


In [ ]:
from pathlib import Path
import json
import torch

CKPT = Path(
    "/content/drive/MyDrive/ROS_VLM_Checkpoints/"
    "E3_recovery_from_step050/step300"
)

assert CKPT.is_dir(), f"Không tìm thấy checkpoint: {CKPT}"
print("Checkpoint:", CKPT)
print("Các file:", [p.name for p in CKPT.iterdir()])

In [ ]:
from peft import PeftModel

model = PeftModel.from_pretrained(
    base,
    str(CKPT),
    is_trainable=False,
)

model.eval()
print("Đã tải LoRA Step 300 thành công.")

In [ ]:
from pathlib import Path
import torch

CKPT = Path(
    "/content/drive/MyDrive/ROS_VLM_Checkpoints/"
    "E3_recovery_from_step050/step300"
)

extra_path = CKPT / "extra_trainable.pt"

extra = torch.load(
    extra_path,
    map_location="cpu",
    weights_only=True,
)

print("Số tensor:", len(extra))
print("\n10 tensor đầu tiên:")

for name, tensor in list(extra.items())[:10]:
    print(
        f"Tên: {name} | "
        f"Shape: {tuple(tensor.shape)} | "
        f"Dtype: {tensor.dtype}"
    )

In [ ]:
import json
from pathlib import Path
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

REPO = Path("/content/K4-Track-4-Day10-VisionLanguageModel")
TEST_FILE = REPO / "data/splits/test.jsonl"

with open(TEST_FILE, "r", encoding="utf-8") as f:
    sample = json.loads(f.readline())

# Tìm ảnh theo đường dẫn trong JSONL
image_path = REPO / sample["image_path"]

if not image_path.exists():
    # Thử tìm ảnh trong thư mục data
    matches = list((REPO / "data").rglob(Path(sample["image_path"]).name))
    if matches:
        image_path = matches[0]
    else:
        raise FileNotFoundError(
            f"Không tìm thấy ảnh: {sample['image_path']}"
        )

image = Image.open(image_path).convert("RGB")
draw = ImageDraw.Draw(image)

W, H = image.size

# YOLO: [center_x, center_y, width, height], giá trị chuẩn hóa 0–1
for obj in sample["objects"]:
    cx, cy, bw, bh = obj["yolo"]

    x1 = int((cx - bw / 2) * W)
    y1 = int((cy - bh / 2) * H)
    x2 = int((cx + bw / 2) * W)
    y2 = int((cy + bh / 2) * H)

    draw.rectangle([x1, y1, x2, y2], outline="red", width=3)
    draw.text((x1, max(0, y1 - 18)), obj["label"], fill="red")

print("Image ID:", sample["image_id"])
print("Đường dẫn ảnh:", image_path)
print("Kích thước:", image.size)
print("Ground truth:", sample["objects"])

plt.figure(figsize=(12, 7))
plt.imshow(image)
plt.axis("off")
plt.show()

In [ ]:
import json
import torch
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

from scripts.labcommon import read_jsonl

REPO = Path("/content/K4-Track-4-Day10-VisionLanguageModel")
DATA_DIR = REPO / "data"
TEST_FILE = DATA_DIR / "splits" / "test.jsonl"
PRED_FILE = REPO / "work" / "predictions" / "E3_step300_test.jsonl"

entries = read_jsonl(TEST_FILE)
PRED_FILE.parent.mkdir(parents=True, exist_ok=True)

model.eval()
results = []

for sample in tqdm(entries, desc="Inference test"):
    image_path = DATA_DIR / sample["image_path"]

    try:
        with Image.open(image_path) as im:
            image = im.convert("RGB")

        messages = [{
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": PROMPT},
            ],
        }]

        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
        ).to(model.device)

        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=192,
                do_sample=False,
            )

        prompt_len = inputs["input_ids"].shape[1]
        raw = processor.batch_decode(
            output[:, prompt_len:],
            skip_special_tokens=True,
        )[0]

    except Exception as e:
        print(f"Lỗi ảnh {sample['image_id']}: {e}")
        raw = ""  # Để bộ chấm ghi nhận dự đoán không hợp lệ

    results.append({
        "image_id": sample["image_id"],
        "raw": raw,
    })

    # Lưu dần để giảm nguy cơ mất kết quả nếu Colab ngắt
    with PRED_FILE.open("a", encoding="utf-8") as f:
        f.write(json.dumps(results[-1], ensure_ascii=False) + "\n")

print(f"Đã xử lý: {len(results)}/{len(entries)} ảnh")
print(f"File dự đoán: {PRED_FILE}")

In [ ]:
from pathlib import Path
import json

REPO = Path("/content/K4-Track-4-Day10-VisionLanguageModel")
pred_file = REPO / "work/predictions/E3_step300_test.jsonl"

rows = [
    json.loads(line)
    for line in pred_file.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

ids = [row["image_id"] for row in rows]

print("Số dòng đã lưu:", len(rows))
print("Số image_id duy nhất:", len(set(ids)))
print("5 dự đoán cuối:")
for row in rows[-5:]:
    print(row["image_id"], "->", row["raw"][:200])

In [ ]:
import json
from pathlib import Path

REPO = Path("/content/K4-Track-4-Day10-VisionLanguageModel")
SPLITS_DIR = REPO / "data" / "splits"
PRED_FILE = REPO / "work" / "predictions" / "E3_step300_test.jsonl"

# Đọc 102 dự đoán đã lưu
preds = [
    json.loads(line)
    for line in PRED_FILE.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
pred_ids = {row["image_id"] for row in preds}

# Lấy đúng 102 ảnh tương ứng từ test.jsonl, giữ nguyên thứ tự của tập test
test_entries = [
    json.loads(line)
    for line in (SPLITS_DIR / "test.jsonl").read_text(encoding="utf-8").splitlines()
    if line.strip()
]
subset = [row for row in test_entries if row["image_id"] in pred_ids]

assert len(subset) == len(preds), (
    f"Không khớp: {len(subset)} ảnh test, {len(preds)} dự đoán"
)

# Tạo thư mục riêng để không sửa dữ liệu gốc
OUT_DIR = REPO / "work" / "scores" / "subset102"
OUT_DIR.mkdir(parents=True, exist_ok=True)

subset_file = OUT_DIR / "test.jsonl"
with subset_file.open("w", encoding="utf-8") as f:
    for row in subset:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Đã tạo tập test con:", len(subset), "ảnh")
print("File:", subset_file)

In [ ]:
from pathlib import Path
import shutil

repo = Path("/content/K4-Track-4-Day10-VisionLanguageModel")

src = repo / "data" / "splits" / "meta.json"
dst = repo / "work" / "scores" / "subset102" / "meta.json"

shutil.copy2(src, dst)

print("Đã sao chép:", dst)
print("File tồn tại:", dst.exists())

In [ ]:
%cd /content/K4-Track-4-Day10-VisionLanguageModel

!python scripts/score.py \
  --pred work/predictions/E3_step300_test.jsonl \
  --splits-dir work/scores/subset102 \
  --split test \
  --coord norm1000 \
  --name E3_step300_subset102 \
  --out work/scores/subset102/result.json \
  --n-boot 1000

In [ ]:

import json
import time
import torch
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

REPO = Path("/content/K4-Track-4-Day10-VisionLanguageModel")
DATA_DIR = REPO / "data"
TEST_FILE = DATA_DIR / "splits" / "test.jsonl"
PRED_FILE = REPO / "work" / "predictions" / "E3_step300_test.jsonl"

# Đọc toàn bộ test
with TEST_FILE.open("r", encoding="utf-8") as f:
    entries = [json.loads(line) for line in f if line.strip()]

# Đọc các dự đoán đã lưu để không chạy trùng
preds = {}
if PRED_FILE.exists():
    with PRED_FILE.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            image_id = row["image_id"]
            if image_id in preds:
                raise ValueError(f"Trùng image_id trong file dự đoán: {image_id}")
            preds[image_id] = row

test_ids = {e["image_id"] for e in entries}
extra_ids = set(preds) - test_ids
if extra_ids:
    print(f"Cảnh báo: có {len(extra_ids)} dự đoán không thuộc tập test.")

# Chỉ xử lý những ảnh chưa có dự đoán
pending = [e for e in entries if e["image_id"] not in preds]

print("Tổng ảnh test:", len(entries))
print("Đã có dự đoán:", len(entries) - len(pending))
print("Còn cần chạy:", len(pending))

PRED_FILE.parent.mkdir(parents=True, exist_ok=True)
model.eval()
start = time.time()
errors = []

with PRED_FILE.open("a", encoding="utf-8") as f:
    for sample in tqdm(pending, desc="Inference phần còn lại"):
        image_path = DATA_DIR / sample["image_path"]

        try:
            with Image.open(image_path) as im:
                image = im.convert("RGB")

            messages = [{
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": PROMPT},
                ],
            }]

            inputs = processor.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_dict=True,
                return_tensors="pt",
            ).to(model.device)

            with torch.inference_mode():
                output = model.generate(
                    **inputs,
                    max_new_tokens=192,
                    do_sample=False,
                )

            prompt_len = inputs["input_ids"].shape[1]
            raw = processor.batch_decode(
                output[:, prompt_len:],
                skip_special_tokens=True,
            )[0]

        except Exception as e:
            errors.append((sample["image_id"], str(e)))
            raw = ""

        row = {"image_id": sample["image_id"], "raw": raw}
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
        f.flush()
        preds[sample["image_id"]] = row

print(f"\nThời gian chạy: {(time.time() - start) / 60:.1f} phút")
print("Số dự đoán đã lưu:", len(preds))
print("Lỗi inference:", len(errors))

if errors:
    print("Một số lỗi đầu tiên:")
    for item in errors[:10]:
        print(item)


## Ghi chú quan trọng
- Không chạy cell huấn luyện gốc của notebook cũ, vì cell đó bắt đầu từ step 1.
- Không xóa `ROS_VLM_Checkpoints/E3_safe_run/step050` hoặc `step100`.
- Nhánh này dùng LoRA từ step050 nhưng projector khởi tạo lại từ model gốc, optimizer/scheduler mới. Báo cáo kết quả phải ghi rõ đây là nhánh recovery, không phải tiếp tục chính xác thí nghiệm gốc.
- Nếu gặp OOM trên T4, dừng runtime và báo lại lỗi; không giảm cấu hình một cách âm thầm vì sẽ thay đổi thí nghiệm.